# 🌿 Plant Disease Classification — Fixed Training Notebook

**Changes from original:**
- ✅ WeightedRandomSampler to handle class imbalance
- ✅ CrossEntropyLoss with class weights
- ✅ Label smoothing to reduce overconfidence
- ✅ Stronger augmentation (RandomResizedCrop, vertical flip, saturation)
- ✅ Cosine LR scheduler
- ✅ Correct model loading (ViT and Swin separated clearly)
- ✅ Per-class accuracy report at end to catch problem classes


In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'hariprasanthu5002'
os.environ['KAGGLE_KEY'] = 'KGAT_87ed6356b64ede123c082703ad93a6b8'


In [ ]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip


In [ ]:
import shutil

shutil.rmtree("/content/plantvillage dataset/grayscale")
shutil.rmtree("/content/plantvillage dataset/segmented")
print("Unused folders removed")


In [ ]:
# ================= CLASS COUNTS (verify imbalance) =================
from collections import defaultdict
import os

dataset_dir = "/content/plantvillage dataset/color"

class_counts = defaultdict(int)
for cls in os.listdir(dataset_dir):
    cls_path = os.path.join(dataset_dir, cls)
    if os.path.isdir(cls_path):
        for file in os.listdir(cls_path):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                class_counts[cls] += 1

total_images = sum(class_counts.values())
print("Total Images:", total_images)
print("\nImages per class (sorted):")
for cls, count in sorted(class_counts.items(), key=lambda x: x[1]):
    print(f"  {cls}: {count}")


In [ ]:
# ================= TRAIN/VAL/TEST SPLIT =================
import os
from sklearn.model_selection import train_test_split

dataset_dir = "/content/plantvillage dataset/color"

image_paths = []
labels = []

for cls in os.listdir(dataset_dir):
    cls_path = os.path.join(dataset_dir, cls)
    if os.path.isdir(cls_path):
        for file in os.listdir(cls_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(cls_path, file))
                labels.append(cls)

X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths, labels, test_size=0.30, stratify=labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))


In [ ]:
# ================= CLASS MAPPINGS =================
class_names = sorted(list(set(labels)))
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

print("Total classes:", len(class_to_idx))

# Save classes.json for inference
import json
with open("classes.json", "w") as f:
    json.dump(class_names, f)
print("classes.json saved.")


In [ ]:
# ================= TRANSFORMS =================
# FIX 1: Stronger augmentation — RandomResizedCrop, vertical flip,
#         stronger ColorJitter. These help the model learn plant leaf
#         features that generalize across Blueberry/Soybean confusion.
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),   # was Resize(224)
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),                        # NEW
    transforms.RandomRotation(20),                          # was 15
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3, saturation=0.3        # was 0.2, 0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [ ]:
# ================= DATASET CLASS =================
from torch.utils.data import Dataset
from PIL import Image

class PlantDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = class_to_idx[self.labels[idx]]
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
# ================= WEIGHTED SAMPLER + DATALOADERS =================
# FIX 2: WeightedRandomSampler — ensures every class gets seen equally
#         often during training regardless of raw image count.
import torch
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight

train_dataset = PlantDataset(X_train, y_train, transform=train_transform)
val_dataset   = PlantDataset(X_val,   y_val,   transform=val_test_transform)
test_dataset  = PlantDataset(X_test,  y_test,  transform=val_test_transform)

# Compute per-class weights
train_label_indices = [class_to_idx[l] for l in y_train]
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=train_label_indices
)
class_weights_tensor = torch.FloatTensor(class_weights_arr)

print("Class weights (sample — higher = rarer class):")
for i, w in enumerate(class_weights_arr):
    print(f"  {class_names[i]}: {w:.3f}")

# Build sample weights for the sampler
sample_weights = [class_weights_arr[class_to_idx[l]] for l in y_train]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

batch_size = 32

# NOTE: Do NOT use shuffle=True when using a sampler
train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                          shuffle=False,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                          shuffle=False,  num_workers=2)

print("\nDataLoaders ready.")


In [ ]:
# ================= TRAINING FUNCTION =================
# Extracted into a reusable function so both ViT and Swin
# use identical training logic.
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def train_model(model, model_name, save_path, num_epochs=10, patience=5):
    model = model.to(device)

    # FIX 3: CrossEntropyLoss with class weights + label_smoothing
    # label_smoothing=0.1 prevents the model from becoming
    # overconfident (e.g., Swin saying 97% Blueberry for a Soybean).
    criterion = nn.CrossEntropyLoss(
        weight=class_weights_tensor.to(device),
        label_smoothing=0.1
    )

    # FIX 4: Differential learning rates — head trains faster,
    #         backbone fine-tunes slowly to keep pretrained features.
    backbone_params = [p for n, p in model.named_parameters()
                       if 'head' not in n and p.requires_grad]
    head_params     = [p for n, p in model.named_parameters()
                       if 'head' in n and p.requires_grad]

    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': 1e-5},
        {'params': head_params,     'lr': 1e-3},
    ], weight_decay=1e-4)

    # FIX 5: Cosine annealing — smoothly reduces LR to avoid sharp
    #         loss spikes late in training.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs
    )

    best_val_acc    = 0
    epochs_no_improve = 0
    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []

    for epoch in range(num_epochs):
        print(f"\n[{model_name}] Epoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # --- TRAIN ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, lbls in tqdm(train_loader):
            images, lbls = images.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted  = torch.max(outputs, 1)
            total        += lbls.size(0)
            correct      += (predicted == lbls).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc  = 100 * correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        # --- VALIDATE ---
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, lbls in tqdm(val_loader):
                images, lbls = images.to(device), lbls.to(device)
                outputs      = model(images)
                loss         = criterion(outputs, lbls)
                val_loss_sum += loss.item()
                _, predicted  = torch.max(outputs, 1)
                val_total    += lbls.size(0)
                val_correct  += (predicted == lbls).sum().item()

        val_loss = val_loss_sum / len(val_loader)
        val_acc  = 100 * val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        scheduler.step()

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            print("Validation improved. Saving model.")
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s)")

        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break

    print(f"\n[{model_name}] Training Complete. Best Val Acc: {best_val_acc:.2f}%")
    model.load_state_dict(torch.load(save_path))
    return model, train_losses, val_losses, train_accs, val_accs


In [ ]:
# ================= TRAIN ViT =================
import torchvision.models as models

print("Setting up ViT-B/16 ...")
vit_model = models.vit_b_16(weights="IMAGENET1K_V1")

# FIX 6: Unfreeze last 6 transformer blocks + head.
# CRITICAL BUG FIXED: previous pattern was f'encoder.layers.{i}'
# which matched nothing because actual torchvision ViT names are
# 'encoder.layers.encoder_layer_6' NOT 'encoder.layers.6'.
# This left only 29K params trainable (bias terms only) — the
# model essentially did not train at all for 10 epochs.
# Correct pattern confirmed from named_parameters() output:
#   encoder.layers.encoder_layer_0.ln_1.weight  <- actual name
#   encoder.layers.encoder_layer_11.mlp.3.bias  <- actual name
# Layers encoder_layer_0 to 5 frozen  -> low-level features preserved
# Layers encoder_layer_6 to 11 trained -> high-level features adapted
# encoder.ln + heads also trained      -> final norm + classifier
for name, param in vit_model.named_parameters():
    if any(f"encoder_layer_{i}" in name for i in [6, 7, 8, 9, 10, 11]):
        param.requires_grad = True
    elif "heads" in name:
        param.requires_grad = True
    elif "encoder.ln" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    len(class_names)
)

# Verify before training — must show ~35M trainable params
total     = sum(p.numel() for p in vit_model.parameters())
trainable = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f"ViT  total params    : {total:,}")
print(f"ViT  trainable params: {trainable:,}  <- expect ~35,000,000")
if trainable < 1_000_000:
    raise RuntimeError("ViT trainable params too low — layer name pattern failed. STOP.")

vit_model, vit_train_losses, vit_val_losses, vit_train_accs, vit_val_accs = train_model(
    vit_model,
    model_name="ViT",
    save_path="best_model.pth",
    num_epochs=10,
    patience=5
)


In [ ]:
# ================= TRAIN Swin =================
print("Setting up Swin-T ...")
swin_model = models.swin_t(weights="IMAGENET1K_V1")

# Swin layer structure confirmed from named_parameters():
#   features.0  -> patch embedding (freeze)
#   features.1  -> stage 1 transformer blocks (freeze)
#   features.2  -> patch merging (freeze)
#   features.3  -> stage 2 transformer blocks (freeze)
#   features.4  -> patch merging (freeze)
#   features.5  -> stage 3 transformer blocks (freeze)
#   features.6  -> patch merging (train)  <- unfreeze from here
#   features.7  -> stage 4 transformer blocks (train)
#   norm        -> final layer norm (train)
#   head        -> classifier (train)
# Previous run had 15M trainable (too many — features.5 was matching
# because 'features.5' is a substring of nothing wrong but the
# norm.weight was also matching 'head' via norm check — fine).
# The real issue was Swin had 15M trainable which is correct for
# features.6+7 — Swin was fine. ViT was the broken one.
for name, param in swin_model.named_parameters():
    if name.startswith("features.6") or name.startswith("features.7"):
        param.requires_grad = True
    elif name.startswith("norm"):
        param.requires_grad = True
    elif name.startswith("head"):
        param.requires_grad = True
    else:
        param.requires_grad = False

swin_model.head = nn.Linear(
    swin_model.head.in_features,
    len(class_names)
)

# Verify before training — must show ~8M trainable params
total     = sum(p.numel() for p in swin_model.parameters())
trainable = sum(p.numel() for p in swin_model.parameters() if p.requires_grad)
print(f"Swin total params    : {total:,}")
print(f"Swin trainable params: {trainable:,}  <- expect ~8,000,000")
if trainable < 1_000_000:
    raise RuntimeError("Swin trainable params too low — layer name pattern failed. STOP.")

swin_model, swin_train_losses, swin_val_losses, swin_train_accs, swin_val_accs = train_model(
    swin_model,
    model_name="Swin",
    save_path="swin-model.pth",
    num_epochs=10,
    patience=5
)


In [ ]:
# ================= PER-CLASS ACCURACY REPORT =================
# This tells you exactly which classes each model still struggles with
# so you can catch Blueberry/Soybean confusion before deploying.
import torch
from collections import defaultdict

def per_class_accuracy(model, loader, model_name):
    model.eval()
    class_correct = defaultdict(int)
    class_total   = defaultdict(int)

    with torch.no_grad():
        for images, lbls in tqdm(loader):
            images, lbls = images.to(device), lbls.to(device)
            outputs      = model(images)
            _, predicted  = torch.max(outputs, 1)

            for true, pred in zip(lbls, predicted):
                cls = idx_to_class[true.item()]
                class_total[cls]   += 1
                class_correct[cls] += int(true == pred)

    print(f"\n[{model_name}] Per-Class Accuracy on Test Set:")
    print(f"{'Class':<55} {'Correct':>7} {'Total':>7} {'Acc':>7}")
    print("-" * 80)
    problem_classes = []
    for cls in sorted(class_total.keys()):
        acc = 100 * class_correct[cls] / class_total[cls]
        flag = " ← LOW" if acc < 85 else ""
        print(f"{cls:<55} {class_correct[cls]:>7} {class_total[cls]:>7} {acc:>6.1f}%{flag}")
        if acc < 85:
            problem_classes.append((cls, acc))

    if problem_classes:
        print(f"\n⚠️  Problem classes for {model_name}:")
        for cls, acc in sorted(problem_classes, key=lambda x: x[1]):
            print(f"   {cls}: {acc:.1f}%")
    else:
        print(f"\n✅ All classes above 85% for {model_name}")

per_class_accuracy(vit_model,  test_loader, "ViT")
per_class_accuracy(swin_model, test_loader, "Swin")


In [ ]:
# ================= TRAINING CURVES =================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Training Curves", fontsize=14)

for ax, losses, title in [
    (axes[0][0], (vit_train_losses,  vit_val_losses),  "ViT — Loss"),
    (axes[0][1], (swin_train_losses, swin_val_losses), "Swin — Loss"),
]:
    ax.plot(losses[0], label="Train")
    ax.plot(losses[1], label="Val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()

for ax, accs, title in [
    (axes[1][0], (vit_train_accs,  vit_val_accs),  "ViT — Accuracy"),
    (axes[1][1], (swin_train_accs, swin_val_accs), "Swin — Accuracy"),
]:
    ax.plot(accs[0], label="Train")
    ax.plot(accs[1], label="Val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()


In [ ]:
# ================= DOWNLOAD MODELS =================
from google.colab import files

files.download('best_model.pth')
files.download('swin-model.pth')
files.download('classes.json')
files.download('training_curves.png')
